In [1]:
import json
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor
from pydantic import BaseModel,Field
from typing import Optional
import openai

In [5]:
client = OpenAI(
    api_key="token-vulcan",  # 输入你的 API Key
    base_url="http://219.147.99.170:40019/v1"
)


class Thought(BaseModel):
    thought: str
    evaluation: Optional[float] = Field(
        description="The evaluation of the thought. It can be a number between 0.1 and 1.0 being 0.1 the worst and 1.0 the best."
    )
def my_tool_function(input_data: Thought):
    # 假设这里是你的自定义逻辑，可以根据input_data来进行一些处理
    return {
        "thought": input_data.thought,
        "evaluation": input_data.evaluation
    }

def qa_base_v2(messages):
    completion = client.beta.chat.completions.parse(
        model="Qwen2-5-14B",
        messages=messages,
        max_tokens=8000,
        response_format="json_object",
        tools=(
            [openai.pydantic_function_tool(my_tool_function)]
        )
    )
    return completion

def qa_base(messages):
    completion = client.chat.completions.create(
        model="Qwen2-5-14B",
        messages=messages,
        logprobs=False,
        # stream=True  # 开启流式输出,
    )
    # return completion
    return completion.choices[0].message.content
    # if compile.status_code == 200:
    #     result = completion[0].choices[0].delta.content
    #     return result
    # else:
    #     print(f"请求失败，状态码: {completion.status_code}")
    #     return ""
    

In [6]:
user_input = "中国的首都是哪里"
input = [{"role": "user", "content": user_input},]
qa_base(input)

'中国的首都是北京。'

In [7]:
TREE_OF_THOUGHTS_SYS_PROMPT = """
You are an expert problem-solving agent designed to not only solve complex problems but also critically evaluate the quality of your thought process and final answers. 
Your task is to follow a structured approach to generate solutions, assess your thoughts, and provide a rating for each on a scale of 0.1 to 1.0. 
This rating should reflect the accuracy and quality of your reasoning and final answer. 
answer in chinese

### Instructions:

1. **Understand the Problem:**
   - Carefully analyze the problem provided by the user.
   - Break down the problem into smaller, manageable parts if necessary.
   - Formulate a clear understanding of the problem before proceeding.

2. **Generate Thoughts:**
   - Create multiple thoughts or steps toward solving the problem.
   - For each thought, document your reasoning, ensuring that it is logical and well-founded.

3. **Self-Evaluation:**
   - After generating each thought, evaluate its accuracy and quality.
   - Assign an evaluation score between 0.1 and 1.0. Use the following guidelines:
     - **0.1 to 0.4:** The thought is flawed, inaccurate, or incomplete.
     - **0.5 to 0.7:** The thought is partially correct but may lack detail or full accuracy.
     - **0.8 to 1.0:** The thought is accurate, complete, and well-reasoned.

4. **Generate Final Answer:**
   - Based on your thoughts, synthesize a final answer to the problem.
   - Ensure the final answer is comprehensive and addresses all aspects of the problem.

5. **Final Evaluation:**
   - Evaluate the overall quality and accuracy of your final answer.
   - Provide a final evaluation score based on the same 0.1 to 1.0 scale.
   
"""

In [19]:
messages = []
system_prompt = TREE_OF_THOUGHTS_SYS_PROMPT
messages.append(
    {"role": "system", "content": system_prompt}
)

task = """

Your task: is to use 4 numbers and basic arithmetic operations (+-*/) to obtain 24 in 1 equation, return only the math

"""

task_2 = """
Your task: is to Write a coherent passage of 4 short paragraphs. The end sentence of each paragraph must be: {input}
"""
task_2_input = task_2.format(input="It isn't difficult to do a handstand if you just stand on your hands. It caught him off guard that space smelled of seared steak. When she didn’t like a guy who was trying to pick her up, she started using sign language. Each person who knows you has a different perception of who you are.")
messages.append(
    {"role": "user", "content": task_2_input}
)

answer = qa_base(messages)

In [20]:
print(answer)

进行手倒立其实并不难，只要你愿意用手支撑身体。他突然发现太空里弥漫着烤肉的气味。当她遇到一些试图搭讪但不喜欢的男士时，她开始用手语回应。每个人对你的认识都是不同的。

### 第一段
进行手倒立其实并不难，只要你愿意用手支撑身体。对于许多初学者来说，手倒立可能看起来像是一项复杂而困难的技巧，但实际上，通过正确的练习和技巧，任何人都可以学会。重要的是保持平衡，找到支撑点，并逐渐增加时间。它并不是一项只有体操运动员才能掌握的技能。

**自我评价：0.9** - 这段文字准确地解释了手倒立的步骤和需要的技巧，同时给出了鼓励和正面的激励。

### 第二段
他突然发现太空里弥漫着烤肉的气味。在太空站内，由于没有空气对流，各种气味可能会异常强烈，而且不易消散。在太空站的实验室内，科学家们常常进行各种实验，其中包括研究食品加工和保存技术。因此，烤肉的气味可能来自这些实验或日常的饮食活动。这让他感到非常惊讶。

**自我评价：0.8** - 这段文字提供了合理的解释，但缺乏具体的科学依据，因此评价为0.8。

### 第三段
当她遇到一些试图搭讪但不喜欢的男士时，她开始用手语回应。手语不仅是一种有效的沟通方式，也是一种有力的自我保护手段。通过使用手语，她可以有效地避开不必要的对话，同时传达出她对当前交流不感兴趣的态度。这不仅保护了她的隐私和安全，还让她能够更加自信地面对那些不受欢迎的搭讪者。

**自我评价：0.9** - 这段文字清楚地解释了为什么她选择使用手语，并提供了合理的理由和细节。

### 第四段
每个人对你的认识都是不同的。人们往往根据不同的经历和背景对他人形成特定的看法。一个朋友可能认为你是个乐于助人的人，而另一人可能觉得你非常独立。这种多样性使得每个人都有独特的视角和看法，因此你的真实面貌可能因人而异。这种多维度的认识也提醒我们要保持开放和包容的态度，以理解和接受不同的观点。

**自我评价：0.9** - 这段文字准确地阐述了每个人对你的不同看法，并提供了合理的解释和建议。

### 总评价
整体而言，这段文字中的各个段落都准确地表达了各自的主题，并且在逻辑性和连贯性上表现良好。最终评价分为0.9。

**总评分：0.9**


In [10]:
number_of_agents = 4
with ThreadPoolExecutor(max_workers=number_of_agents) as executor:
    next_thoughts = list(
                executor.map(qa_base, [messages] * number_of_agents)
            )

In [11]:
print(len(next_thoughts))
next_thoughts[0]

4


'### 理解问题：\n问题要求使用4个数字和基本的算术运算（+-*/）来得到24，并且只需返回数学表达式。\n\n### 生成思路：\n1. **思路1**：假设数字为1, 2, 3, 4。尝试不同的组合和运算来达到24。\n   - 推理：可以先尝试简单的组合，比如 (1+2+3)*4。\n   - 评分：0.8\n\n2. **思路2**：假设数字为6, 6, 6, 6。尝试不同的组合和运算来达到24。\n   - 推理：可以先尝试简单的组合，比如 6*6-6*6 或 6+6+6+6，但这些都不行。我们需要更复杂一些，如 (6/6+6)*6。\n   - 评分：0.7\n\n3. **思路3**：假设数字为8, 8, 3, 3。尝试不同的组合和运算来达到24。\n   - 推理：可以尝试 (8+8)*(3-3/3)。\n   - 评分：0.9\n\n### 自我评估：\n1. 思路1：(1+2+3)*4 = 24\n   - 评分：0.8\n\n2. 思路2：(6/6+6)*6 = 24\n   - 评分：0.7\n\n3. 思路3：(8+8)*(3-3/3) = 24\n   - 评分：0.9\n\n### 最终答案：\n选择思路3的表达式作为最终答案：(8+8)*(3-3/3) = 24。\n\n### 最终评估：\n最终答案是准确且完整的，满足题目要求。\n- 评分：1.0'

In [12]:
for idx,one_thought in enumerate(next_thoughts):
    print("#####index:",idx)
    print(one_thought)

#####index: 0
### 理解问题：
问题要求使用4个数字和基本的算术运算（+-*/）来得到24，并且只需返回数学表达式。

### 生成思路：
1. **思路1**：假设数字为1, 2, 3, 4。尝试不同的组合和运算来达到24。
   - 推理：可以先尝试简单的组合，比如 (1+2+3)*4。
   - 评分：0.8

2. **思路2**：假设数字为6, 6, 6, 6。尝试不同的组合和运算来达到24。
   - 推理：可以先尝试简单的组合，比如 6*6-6*6 或 6+6+6+6，但这些都不行。我们需要更复杂一些，如 (6/6+6)*6。
   - 评分：0.7

3. **思路3**：假设数字为8, 8, 3, 3。尝试不同的组合和运算来达到24。
   - 推理：可以尝试 (8+8)*(3-3/3)。
   - 评分：0.9

### 自我评估：
1. 思路1：(1+2+3)*4 = 24
   - 评分：0.8

2. 思路2：(6/6+6)*6 = 24
   - 评分：0.7

3. 思路3：(8+8)*(3-3/3) = 24
   - 评分：0.9

### 最终答案：
选择思路3的表达式作为最终答案：(8+8)*(3-3/3) = 24。

### 最终评估：
最终答案是准确且完整的，满足题目要求。
- 评分：1.0
#####index: 1
### 理解问题：
我们需要使用4个数字和基本的算术运算（+-*/）来得到24，且结果必须是一个表达式。

### 生成思路：
1. **思路1：**
   - 使用数字1, 2, 3, 4，尝试不同的组合和运算。
   - 例如：\( (1 + 2 + 3) \times 4 \)

   **推理：**
   - \( (1 + 2 + 3) = 6 \)
   - \( 6 \times 4 = 24 \)

   **评估：**
   - 这个思路是正确的，可以得到24。
   - 评分：0.8

2. **思路2：**
   - 使用数字1, 3, 8, 12，尝试不同的组合和运算。
   - 例如：\( 12 \div (1 + 3) \times 8 \)

   **推理：**
   - \( (1 + 3) = 4 \)
   - \( 12 \div 4 = 3 \)
 

In [13]:
messages = []
system_prompt = TREE_OF_THOUGHTS_SYS_PROMPT
messages.append(
    {"role": "system", "content": system_prompt}
)
messages.append(
    {"role": "user", "content": next_thoughts[-1]}
)

answer = qa_base(messages)

In [14]:
print(answer)

### 理解问题
问题要求使用4个数字和基本的算术运算（+-*/）来得到24，并且只需要返回算术表达式。

### 生成思路
1. **尝试不同的组合**：
   - 例如，假设给定的数字是1, 2, 3, 4。
   - 尝试不同的运算组合，如 (1+2+3+4)、(1*2*3*4)、(4*6) 等等。

2. **具体计算**：
   - 试试 (4 * (3 + (2 + 1)))。

### 自我评估
- **思路1**：
  - 尝试不同的组合是合理的，但没有具体计算，只是一种尝试。
  - 评分：0.5

- **思路2**：
  - 具体计算 (4 * (3 + (2 + 1))) 是一种明确的方法，可以验证其结果。
  - 评分：0.8

### 最终答案
计算 (4 * (3 + (2 + 1)))：
- (2 + 1) = 3
- 3 + 3 = 6
- 4 * 6 = 24

最终答案：`4 * (3 + (2 + 1))`

### 最终评估
该答案直接且正确，满足题目要求。
- 评分：1.0

总结：此解决方案思路清晰，计算准确，最终答案达到了题目要求。
